# Notebook 01: Setup and Baseline Test

**Agency Calculus Empirical Validation — Paper C**

This notebook:
1. Installs the AI Economist (Salesforce Foundation) and dependencies
2. Verifies the simulation environment works
3. Runs a short baseline training (100K steps) to confirm RLlib integration

Target platforms: Google Colab, Kaggle

## 1. Detect Platform & Install Dependencies

In [ ]:
import sys, os

# Detect platform
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
print(f'Colab: {IN_COLAB}, Kaggle: {IN_KAGGLE}')

In [ ]:
# Install AI Economist from GitHub
!pip install -q git+https://github.com/salesforce/ai-economist.git

# Install RLlib (Ray 2.3 is last version with good gym 0.21 compatibility)
!pip install -q 'ray[rllib]==2.3.0' torch

# Other deps
!pip install -q matplotlib seaborn pandas scipy tqdm

In [ ]:
# Clone ac-validation repo (skip if already present)
import os
if not os.path.exists('ac-validation'):
    !git clone https://github.com/caseymrobbins/ac-validation.git

# Add src to path
sys.path.insert(0, 'ac-validation/src')
# Or if running from repo root:
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

## 2. Verify AI Economist Environment

In [ ]:
import ai_economist
print(f'AI Economist version: {ai_economist.__version__}')

from ai_economist.foundation.base.base_env import BaseEnvironment
from ai_economist import foundation
print('Foundation import OK')

In [ ]:
# Standard AI Economist environment configuration
# 4 worker agents + 1 planner, small map for quick testing

env_config = {
    'scenario_name': 'simple_wood_and_stone/simple_wood_and_stone',
    'components': [
        {'Build': {'skill_dist': 'pareto', 'payment_max_skill_multiplier': 3}},
        {'ContinuousDoubleAuction': {'max_num_orders': 5}},
        {'Gather': {}},
    ],
    'env_layout_file': 'quadrant_25x25_20each_30clump.txt',
    'starting_agent_coin': 10,
    'n_agents': 4,
    'world_size': [25, 25],
    'episode_length': 1000,
    'multi_action_mode_agents': False,
    'multi_action_mode_planner': True,
    'flatten_observations': False,
    'flatten_masks': True,
    'allow_observation_scaling': True,
    'mixing_weight_gini_vs_coin': 0.0,  # We override planner reward ourselves
}

In [ ]:
# Create environment
env = foundation.make_env_instance(**env_config)
print(f'Environment created: {type(env).__name__}')
print(f'n_agents: {env.n_agents}')
print(f'World size: {env.world_size}')

obs = env.reset()
print(f'Observation keys: {list(obs.keys())}')
print(f'Agent 0 obs keys: {list(obs["0"].keys()) if isinstance(obs.get("0"), dict) else "flat"}')

In [ ]:
# Run a few random steps to confirm env is working
import numpy as np

obs = env.reset()
total_rewards = {k: 0.0 for k in obs.keys()}

for step in range(100):
    actions = {}
    for agent_id in range(env.n_agents):
        # Random action from agent's action space
        agent = env.get_agent(str(agent_id))
        actions[str(agent_id)] = {
            action_name: np.random.randint(0, n_actions)
            for action_name, n_actions in agent.action_spaces.items()
        }
    # Random planner action
    planner = env.get_agent('p')
    actions['p'] = {
        action_name: np.random.randint(0, n_actions)
        for action_name, n_actions in planner.action_spaces.items()
    }
    
    obs, rewards, done, info = env.step(actions)
    for k, r in rewards.items():
        total_rewards[k] = total_rewards.get(k, 0) + r

print('100 random steps completed')
print('Cumulative rewards (random policy):')
for k, r in sorted(total_rewards.items()):
    print(f'  Agent {k}: {r:.2f}')

## 3. Verify POLI + Metrics Modules

In [ ]:
from ac_rewards import sum_reward, nash_reward, jam_reward, get_reward_fn
from poli_agency import compute_agency, geometric_mean
from metrics import gini_coefficient, compute_episode_metrics, MetricsLogger

# Quick smoke tests
utils = [10.0, 5.0, 8.0, 2.0]

print(f'SUM reward:  {sum_reward(utils):.4f}  (expected 25.0)')
print(f'NASH reward: {nash_reward(utils):.4f}')
print(f'JAM reward:  {jam_reward(utils):.4f}  (expected log(2) = {np.log(2):.4f})')

# JAM with zero utility — should return -1e10
print(f'JAM with zero: {jam_reward([0.0, 5.0, 8.0, 2.0])}')

# POLI
agency = compute_agency(
    coin=50.0, inventory={'wood': 5, 'stone': 3},
    reachable_tiles=30, tradeable_goods=2,
    available_actions=12, income_change=5.0,
    observable_fraction=0.3,
)
print(f'\nPOLI agency score: {agency["agency"]:.4f}')
print(f'  P={agency["prerequisites"]:.3f} O={agency["options"]:.3f} '
      f'L={agency["levers"]:.3f} I={agency["impact"]:.3f} K={agency["knowledge"]:.3f}')

# Gini
equal = [10, 10, 10, 10]
unequal = [1, 1, 1, 97]
print(f'\nGini (equal):   {gini_coefficient(equal):.4f}  (expected 0.0)')
print(f'Gini (unequal): {gini_coefficient(unequal):.4f}')

## 4. Short Baseline RLlib Training (100K steps)

This confirms RLlib + AI Economist integration works before starting the full experiment.
Expected runtime: ~5-10 minutes on Colab CPU.

In [ ]:
import ray
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.env.wrappers.multi_agent_env_compatibility import MultiAgentEnvCompatibility

# Wrap AI Economist for RLlib
# The AI Economist has its own RLlib integration — use it directly
try:
    from ai_economist.training.rllib_wrapper import RLlibEnvWrapper
    print('Using ai_economist RLlibEnvWrapper')
    USE_RLLIB_WRAPPER = True
except ImportError:
    print('RLlibEnvWrapper not found, using gym wrapper')
    USE_RLLIB_WRAPPER = False

In [ ]:
# Initialize Ray
if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, num_cpus=2, log_to_driver=False)
print(f'Ray initialized: {ray.is_initialized()}')

In [ ]:
# Minimal training config for baseline verification
# Full training configs are in notebooks 04-06

if USE_RLLIB_WRAPPER:
    # AI Economist training script approach
    # See: https://github.com/salesforce/ai-economist/blob/master/tutorials/multi_agent_training_with_rllib.ipynb
    
    trainer_config = {
        'env': RLlibEnvWrapper,
        'env_config': {
            'env_config_dict': env_config,
            'num_envs_per_worker': 1,
        },
        'num_workers': 1,
        'num_gpus': 0,
        'train_batch_size': 4000,
        'rollout_fragment_length': 200,
        'framework': 'torch',
    }
    
    from ray.rllib.algorithms.ppo import PPO
    trainer = PPO(config=trainer_config)
    
    print('Training for 1 iteration (~4000 steps) to verify...')
    result = trainer.train()
    print(f"Iteration 1 complete")
    print(f"  episode_reward_mean: {result.get('episode_reward_mean', 'N/A')}")
    print(f"  timesteps_total: {result.get('timesteps_total', 'N/A')}")
    print('\nBaseline RLlib integration: OK')
    trainer.stop()

else:
    # Fallback: just confirm env step loop works at scale
    print('Running 10K environment steps (no RLlib)...')
    obs = env.reset()
    for step in range(10000):
        actions = {}
        for agent_id in range(env.n_agents):
            agent = env.get_agent(str(agent_id))
            actions[str(agent_id)] = {
                k: np.random.randint(0, v)
                for k, v in agent.action_spaces.items()
            }
        planner = env.get_agent('p')
        actions['p'] = {
            k: np.random.randint(0, v)
            for k, v in planner.action_spaces.items()
        }
        obs, rewards, done, info = env.step(actions)
        if done.get('__all__', False):
            obs = env.reset()
    print('10K steps completed. Environment: OK')

## 5. Summary

- AI Economist: installed and verified
- Environment: creates, resets, steps correctly
- POLI/Metrics/Rewards modules: imported and smoke-tested
- RLlib integration: baseline training completed

**Next:** Notebook 02 — implement and test SUM/NASH/JAM reward functions in the AI Economist planner.

In [ ]:
# Cleanup
if ray.is_initialized():
    ray.shutdown()
print('Setup complete. Ready for notebook 02.')